# KINO Renderingssession

Gratis T4 i Colab (eller T4/P100 på Kaggle). Notebooken **anrokar KINO** och hämtar jobb — ingen tunnel, ingen ngrok.

1. I KINO: Inställningar → **Starta renderingssession** och kopiera koden.
2. Klistra in koden i cellen nedan och kör alla celler.
3. Gå tillbaka till KINO och rendera. Statusen ska visa t.ex. **GPU-session: Colab T4 aktiv**.

Sessionen slutar när du stänger notebooken, efter några timmars gratis-GPU, eller efter ca 60 s utan kontakt (då faller KINO tillbaka på Hugging Face-kedjan).

**Kaggle:** ett konto per person, för personligt bruk. Internet måste vara på, accelerator GPU T4 (eller P100).


**Uppskattade tider per klipp (inte mätta här — den här miljön har ingen T4).** Siffrorna är ungefärliga, utifrån modellstorlek, 16 GB-kort och vanliga rapporter för liknande pipelines:

- **LTX-Video 2B**, 480p, ca 5 s: typiskt **45–120 s** per klipp på T4 (LTX är förhållandevis snabb).
- **Wan 2.1 1.3B**, 480p, ca 5 s: typiskt **1–3 min** per klipp på T4.
- **Wan 2.2 TI2V-5B** med CPU-offload, 480p, ca 5 s: typiskt **3–8 min** per klipp på T4 (vikterna + överföring till CPU gör den långsammare).
- **MuseTalk 1.5** (äkta läppsynk, MIT): bara munnen/käken ritas om, fp16, ca 4–6 GB VRAM. Första gången hämtas kod + vikter (~4 GB, typiskt **3–8 min**). Sedan ungefär **1–3 min** för 5 s på T4 (ansiktssökning + MuseTalk).
- **Levande bild + läppsynk**: först LTX-Video (bilden får rörelse, ca 4 s, förlängs fram-och-tillbaka om ljudet är längre), sedan MuseTalk. Typiskt **4–10 min** per klipp på T4, första gången längre.
- **WhisperX** (Klipp + textremsor ord för ord, BSD-2): installeras i en egen Python-miljö i bakgrunden direkt efter parningen (första gången ca 5–10 min inkl. large-v3 och svensk/engelsk ordjustering, ~5 GB). Sedan typiskt **1–3 min per 10 min tal** på T4 (fp16). Talare känns igen med SpeechBrain ECAPA (inte pyannote, som kräver HF-token).
- **Qwen3.8 27B abliterated** (bildförståelse + manus, *inte* bildgenerering): llama.cpp med GGUF från huihui-ai + mmproj för syn. 2×T4 (Kaggle) → UD-Q4_K_XL (17,4 GB, delas på båda korten); 1×T4 (Colab) → UD-IQ3_XXS (11 GB). Första gången hämtas ~12–18 GB (typiskt **3–10 min**) och på äldre Ubuntu byggs llama.cpp för T4 (**~10–20 min**, en gång per session). Laddning ~1–3 min, sedan ungefär **15–60 s** per bildbeskrivning och **2–5 min** för ett längre manus. Laddas ur automatiskt innan video/läppsynk/transkribering och laddas igen vid nästa fråga. Stäng av med `QWEN38 = False`.
- **Kandinsky S2V Lite** (läppsynk) körs **inte** på T4/P100: dokumentationen anger ungefär 28 GB även med VAE-tiling, alltså över 16 GB.

Licenser: Wan Apache-2.0; Qwen3.8 27B Apache-2.0, llama.cpp MIT; WhisperX BSD-2 (faster-whisper MIT, wav2vec2-justeringsmodeller per språk, SpeechBrain Apache-2.0); MuseTalk MIT (vikter: sd-vae-ft-mse, whisper-tiny, BiSeNet face parsing); LTX-Video enligt Lightricks community license / AUP (bara SFW — workern skickar inga NSFW-jobb).


### 1. Starta GPU och hämta workern


In [ ]:
import os
try:
    from google.colab import drive  # noqa: F401
    print('Colab OK')
except Exception:
    print('Inte Colab — kör cellerna ändå om du har GPU.')


In [ ]:
import os, sys, urllib.request, shutil, subprocess
from pathlib import Path

# Hämta senaste workern från repot; fall tillbaka på en lokal kopia om nätet strular.
RAW = "https://raw.githubusercontent.com/pistpejl/kino/main/notebooks/kino_worker.py"
MODELS = "https://raw.githubusercontent.com/pistpejl/kino/main/notebooks/kino_worker_models.py"
MUSETALK = "https://raw.githubusercontent.com/pistpejl/kino/main/notebooks/kino_musetalk.py"
QWEN38_URL = "https://raw.githubusercontent.com/pistpejl/kino/main/notebooks/kino_qwen38.py"
WHISPERX = "https://raw.githubusercontent.com/pistpejl/kino/main/notebooks/kino_whisperx.py"
root = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
for url, name in ((RAW, "kino_worker.py"), (MODELS, "kino_worker_models.py"), (MUSETALK, "kino_musetalk.py"), (WHISPERX, "kino_whisperx.py", (QWEN38_URL, "kino_qwen38.py"))):
    dest = root / name
    try:
        urllib.request.urlretrieve(url, dest)
        print("hämtade", name)
    except Exception as e:
        print("kunde inte hämta", name, e)
sys.path.insert(0, str(root))
print("redo", root)


In [ ]:
# Paket för riktiga modeller (hoppas över i mock-läge). T4/sm75: inga flash-attn/fp8-hjul.
import os
if os.environ.get("KINO_WORKER_MOCK") == "1":
    print("Mock-läge: hoppar över modellpaket")
else:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers", "transformers", "accelerate", "safetensors", "pillow", "imageio", "imageio-ffmpeg",
                           # MuseTalk 1.5 (läppsynk). Vikterna (~4 GB) hämtas först när första läppsynken körs.
                           "librosa", "einops", "omegaconf", "soundfile", "face-alignment", "huggingface_hub"])
    print("Paket installerade")


### 2. Klistra in koden och 3. Rendera i KINO


In [ ]:
# 2. Klistra in koden från KINO här:
PAIR_CODE = ""  # t.ex. AB12-CD34
KINO_SERVER = "https://grok-musik-video.onrender.com"

# Låt stå False för riktig GPU-rendering. True = bara en ffmpeg-testloop (ingen modell).
MOCK = False

# Qwen3.8 27B (bildförståelse + manus via llama.cpp). False = hoppa över nedladdningen (~12–18 GB).
QWEN38 = True

import os, sys, subprocess
from pathlib import Path
if not PAIR_CODE.strip():
    raise SystemExit("Klistra in PAIR_CODE från KINO (Starta renderingssession) först.")

worker = None
for p in (Path("/content/kino_worker.py"), Path("/kaggle/working/kino_worker.py"), Path("kino_worker.py")):
    if p.exists():
        worker = p
        break
if worker is None:
    raise SystemExit("kino_worker.py saknas — kör cellen som hämtar workern först.")

os.environ["KINO_QWEN38"] = "1" if QWEN38 else "0"
cmd = [sys.executable, "-u", str(worker), "--server", KINO_SERVER, "--code", PAIR_CODE.strip()]
if MOCK:
    cmd.append("--mock")
print("Startar worker mot", KINO_SERVER)
# Blockerar så länge sessionen ska leva. Stoppa cellen för att avsluta.
subprocess.check_call(cmd)
